# Notebook 08: Regional Video Game Market Preferences

## Objective

This notebook examines how video game sales patterns differ across North America, Europe, and Japan.

Previous analyses primarily evaluated overall performance using global sales, game characteristics, and predictive modeling. However, global totals can hide important differences between regional markets. A genre that performs strongly in one region may have limited popularity in another.

This analysis compares genre-level sales across the three main regional markets in the dataset:

- North America
- Europe
- Japan

Both total regional sales and proportional sales distributions will be examined. Using proportions is important because the regional markets differ substantially in total sales volume.

The results will help identify genres with broad international appeal and genres whose commercial performance is concentrated in a particular region.

## Research Question

**How do video game genre preferences differ across North America, Europe, and Japan?**

This analysis will answer the following supporting questions:

1. Which genres generate the most sales in each region?
2. What percentage of each region's total sales is generated by each genre?
3. Which genres have similar popularity across regions?
4. Which genres are particularly concentrated in one regional market?

In [5]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Visualization settings
sns.set_theme(style="whitegrid")

In [6]:
candidate_paths = [
    Path("../../data/processed/video_game_sales_cleaned.csv"),
    Path("../data/processed/video_game_sales_cleaned.csv"),
    Path("data/processed/video_game_sales_cleaned.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find video_game_sales_cleaned.csv. "
        "Update candidate_paths to match the notebook's location."
    )

df = pd.read_csv(data_path)

print(f"Loaded: {data_path.resolve()}")
print(f"Dataset shape: {df.shape}")
df.head()

Loaded: C:\Users\tyler\OneDrive\Desktop\26projects\data_an\data-analysis-on-the-videogame-industry\data\processed\video_game_sales_cleaned.csv
Dataset shape: (16287, 19)


,name,platform,year,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales,regional_sales_total,decade,sales_tier,is_high_seller,na_sales_share,eu_sales_share,jp_sales_share,other_sales_share,dominant_region
0,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,82.74,2000s,blockbuster,1,0.50,0.35,0.05,0.10,North America
1,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,40.24,1980s,blockbuster,1,0.72,0.09,0.17,0.02,North America
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,35.83,2000s,blockbuster,1,0.44,0.36,0.11,0.09,North America
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,33.00,2000s,blockbuster,1,0.48,0.33,0.10,0.09,North America
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,31.38,1990s,blockbuster,1,0.36,0.28,0.33,0.03,North America


In [7]:
# Columns required for the regional genre analysis
regional_columns = [
    "genre",
    "na_sales",
    "eu_sales",
    "jp_sales"
]

regional_df = df[regional_columns].copy()

regional_df.head()

,genre,na_sales,eu_sales,jp_sales
0,Sports,41.49,29.02,3.77
1,Platform,29.08,3.58,6.81
2,Racing,15.85,12.88,3.79
3,Sports,15.75,11.01,3.28
4,Role-Playing,11.27,8.89,10.22


In [8]:
# Confirm that regional sales columns contain valid numeric values
regional_df[["na_sales", "eu_sales", "jp_sales"]].describe()

,na_sales,eu_sales,jp_sales
count,"16,287.00","16,287.00","16,287.00"
mean,0.27,0.15,0.08
std,0.82,0.51,0.31
min,0.00,0.00,0.00
25%,0.00,0.00,0.00
50%,0.08,0.02,0.00
75%,0.24,0.11,0.04
max,41.49,29.02,10.22


In [9]:
# Regional sales should not contain negative values
negative_sales = (
    regional_df[["na_sales", "eu_sales", "jp_sales"]] < 0
).sum()

negative_sales

na_sales    0
eu_sales    0
jp_sales    0
dtype: int64

## 1. Total Sales by Genre and Region

The first stage of the analysis aggregates total sales for each genre within North America, Europe, and Japan.

These totals show the commercial size of each genre in each region. However, because the three regional markets have different total sales volumes, market-share percentages will also be calculated later for a more comparable measure of regional preference.

In [10]:
# Aggregate regional sales by genre
genre_regional_sales = (
    regional_df
    .groupby("genre", as_index=False)[
        ["na_sales", "eu_sales", "jp_sales"]
    ]
    .sum()
)

# Add combined sales across the three selected regions
genre_regional_sales["combined_regional_sales"] = (
    genre_regional_sales["na_sales"]
    + genre_regional_sales["eu_sales"]
    + genre_regional_sales["jp_sales"]
)

# Sort genres by combined regional sales
genre_regional_sales = genre_regional_sales.sort_values(
    "combined_regional_sales",
    ascending=False
).reset_index(drop=True)

genre_regional_sales

,genre,na_sales,eu_sales,jp_sales,combined_regional_sales
0,Action,861.77,516.48,158.64,"1,536.89"
1,Sports,670.09,371.34,134.76,"1,176.19"
2,Shooter,575.16,310.45,38.18,923.79
3,Role-Playing,326.50,187.57,350.25,864.32
4,Platform,445.99,200.65,130.65,777.29
5,Misc,396.92,211.77,106.67,715.36
6,Racing,356.93,236.31,56.61,649.85
7,Fighting,220.74,100.00,87.15,407.89
8,Simulation,181.51,113.02,63.54,358.07
9,Puzzle,122.01,50.52,56.68,229.21
